In [18]:
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableParallel
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

In [19]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

In [20]:
prompt = ChatPromptTemplate.from_template(
    "Explain the topic: {query}"
)

In [21]:
llm = ChatOpenAI(
    api_key=api_key,
    model="gpt-4o-mini"
)

In [22]:
parser = StrOutputParser()

In [23]:
clean_query = RunnableLambda(
    lambda x: x["query"].strip().lower()
)

In [24]:
input_stage = RunnablePassthrough.assign(
    cleaned_query=clean_query
)

In [25]:
main_chain = (
    {"query": lambda x: x["cleaned_query"]}  # use cleaned query
    | prompt 
    | llm
    | parser
)

In [26]:

word_count = RunnableLambda(
    lambda text: len(text.split())
)

In [27]:

final_chain = (
    input_stage
    | RunnableParallel(
        answer=main_chain,
        words=lambda x: word_count.invoke(
            main_chain.invoke(x)
        )
    )
)

In [28]:
result = final_chain.invoke({
    "query": "   Explain AI Agents   "
})

print(result)

{'answer': "AI agents are systems or programs designed to perform tasks autonomously or with minimal human intervention using artificial intelligence techniques. They can operate in a variety of environments and domains, making decisions, learning from experiences, and adapting to changing conditions. Here's a breakdown of some key concepts related to AI agents:\n\n### Types of AI Agents\n\n1. **Reactive Agents**:\n   - These agents operate based on current perceptions and do not maintain an internal state or memory of past actions. They respond to stimuli in their environment using pre-defined rules. A simple example is an AI that plays a game by reacting to the opponent’s moves.\n\n2. **Deliberative Agents**:\n   - These agents maintain an internal model of the world, allowing them to plan their actions based on both current states and long-term goals. They involve reasoning and problem-solving, making decisions based on predictions and expected outcomes.\n\n3. **Learning Agents**:\n